# Регрессия CC50

Задача: предсказать концентрацию полумаксимальной цитотоксичности CC50 по молекулярным дескрипторам.

## 1. Импорты

In [1]:
import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit, RandomizedSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 2. Загрузка и подготовка данных

Загружаем данные и применяем решения из EDA: убираем технический индекс, удаляем дубликаты строк.

In [2]:
raw_data = pd.read_excel("../data/dataset.xlsx")

TARGET_COLUMNS = ["IC50, mM", "CC50, mM", "SI"]
TARGET = "CC50, mM"

data = raw_data.iloc[:, 1:].copy()
data = data.drop_duplicates()

feature_columns = [col for col in data.columns if col not in TARGET_COLUMNS]
X = data[feature_columns]
y = data[TARGET]

print(f"Объектов: {len(X)}, признаков: {X.shape[1]}")
print(f"CC50: min={y.min():.2f}, median={y.median():.2f}, max={y.max():.2f} mM")
print(f"Пропусков в признаках: {X.isna().sum().sum()}")

Объектов: 969, признаков: 210
CC50: min=0.70, median=424.17, max=4538.98 mM
Пропусков в признаках: 36


## 3. Разбиение на обучение и тест

Делим данные на обучение и тест с помощью группового разбиения. Объекты с одинаковыми значениями молекулярных дескрипторов относим к одной группе, чтобы они не могли одновременно попасть в обучение и тест.

Для подбора гиперпараметров используем групповую кросс-валидацию. Целевую переменную преобразуем с помощью log1p, так как распределение CC50 скошено вправо.

In [3]:
groups = pd.util.hash_pandas_object(X, index=False)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
cv = GroupKFold(n_splits=5)

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(f"Обучение: {len(X_train)} объектов, тест: {len(X_test)} объектов")

Обучение: 759 объектов, тест: 210 объектов


## 4. Как измеряем качество

- MAE (Mean Absolute Error) - средний модуль ошибки в mM. Показывает, на сколько в среднем модель ошибается.
- RMSE (Root Mean Squared Error) - корень из среднего квадрата ошибки. Сильнее наказывает за большие промахи - если RMSE заметно больше MAE, значит на каких-то объектах модель ошибается особенно сильно.

Обе метрики считаем на исходной шкале (mM): модель предсказывает логарифм, ответ переводим обратно через expm1 и сравниваем с истиной уже в mM.

При подборе гиперпараметров через кросс-валидацию используем MAE на лог-шкале.

### Базовая модель

Сначала проверим простейшую модель, которая для всех объектов предсказывает медианное значение обучающей выборки. Это позволит понять, действительно ли обученные модели находят зависимости в данных.

In [4]:
dummy_model = DummyRegressor(strategy="median")
dummy_model.fit(X_train, y_train_log)

dummy_pred_log = dummy_model.predict(X_test)
dummy_pred = np.expm1(dummy_pred_log)

dummy_mae = mean_absolute_error(y_test, dummy_pred)
dummy_rmse = np.sqrt(mean_squared_error(y_test, dummy_pred))

print(
    f"Baseline | Test MAE: {dummy_mae:.2f} mM | "
    f"Test RMSE: {dummy_rmse:.2f} mM"
)

Baseline | Test MAE: 475.62 mM | Test RMSE: 694.26 mM


## 5. Ridge (L2-регуляризация)

Ridge добавляет к ошибке штраф за сумму квадратов коэффициентов, умноженную на параметр alpha. Штраф не даёт коэффициентам раздуваться, что важно при скоррелированных признаках. Чем больше alpha, тем сильнее коэффициенты "прижаты" к нулю.

Признаки предварительно масштабируем через StandardScaler, чтобы штраф действовал равномерно.

In [5]:
ridge_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    Ridge(random_state=42),
)

ridge_params = {"ridge__alpha": [0.1, 1, 10, 100, 200, 500, 1000]}

ridge_search = GridSearchCV(
    ridge_pipe, ridge_params,
    cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1,
)
ridge_search.fit(X_train, y_train_log, groups=groups_train)

ridge_cv_results = pd.DataFrame({
    "alpha": ridge_params["ridge__alpha"],
    "CV MAE (лог-шкала)": -ridge_search.cv_results_["mean_test_score"],
})
display(ridge_cv_results)

ridge_best = ridge_search.best_estimator_
print(f"Лучший alpha: {ridge_search.best_params_['ridge__alpha']}")
print(f"CV MAE (лог-шкала): {-ridge_search.best_score_:.4f}")

,alpha,CV MAE (лог-шкала)
0,0.1,1.121275
1,1.0,1.048430
2,10.0,1.011679
3,100.0,1.001123
4,200.0,0.998653
5,500.0,1.008096
6,1000.0,1.041971


Лучший alpha: 200
CV MAE (лог-шкала): 0.9987


Посмотрим на тестовой выборке.

In [6]:
y_pred_log = ridge_best.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test_log)

ridge_mae = mean_absolute_error(y_true, y_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Ridge | Test MAE: {ridge_mae:.2f} mM | Test RMSE: {ridge_rmse:.2f} mM")

Ridge | Test MAE: 389.20 mM | Test RMSE: 638.71 mM


MAE модели составляет 389.20 mM, RMSE — 638.71 mM. Более высокое значение RMSE показывает, что на отдельных объектах модель допускает крупные ошибки.

## 6. Lasso (L1-регуляризация)

Lasso штрафует сумму модулей коэффициентов. Главное отличие от Ridge: L1-штраф может занулить коэффициент целиком, убирая признак из модели. Полезно, когда много избыточных дескрипторов.

In [7]:
lasso_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    Lasso(random_state=42, max_iter=20000),
)

lasso_params = {"lasso__alpha": [0.001, 0.01, 0.1, 1]}

lasso_search = GridSearchCV(
    lasso_pipe, lasso_params,
    cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1,
)
lasso_search.fit(X_train, y_train_log, groups=groups_train)

lasso_cv_results = pd.DataFrame({
    "alpha": lasso_params["lasso__alpha"],
    "CV MAE (лог-шкала)": -lasso_search.cv_results_["mean_test_score"],
})
display(lasso_cv_results)

lasso_best = lasso_search.best_estimator_
n_total = X.shape[1]
n_nonzero = (lasso_best.named_steps["lasso"].coef_ != 0).sum()
print(f"Лучший alpha: {lasso_search.best_params_['lasso__alpha']}")
print(f"Занулено признаков: {n_total - n_nonzero} из {n_total}")
print(f"CV MAE (лог-шкала): {-lasso_search.best_score_:.4f}")

,alpha,CV MAE (лог-шкала)
0,0.001,1.048163
1,0.010,0.995077
2,0.100,1.058744
3,1.000,1.272185


Лучший alpha: 0.01
Занулено признаков: 120 из 210
CV MAE (лог-шкала): 0.9951


Посмотрим на тесте.

In [8]:
y_pred_log = lasso_best.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test_log)

lasso_mae = mean_absolute_error(y_true, y_pred)
lasso_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Lasso | Test MAE: {lasso_mae:.2f} mM | Test RMSE: {lasso_rmse:.2f} mM")

Lasso | Test MAE: 385.13 mM | Test RMSE: 624.28 mM


MAE модели составляет 385.13 mM, что немного лучше результата Ridge. Lasso занулил 120 коэффициентов из 210, однако это не означает, что соответствующие признаки не нужны: результат может зависеть от корреляции признаков и выбранного разбиения данных.

## 7. K ближайших соседей

KNN запоминает обучающие примеры и для нового объекта усредняет ответы K ближайших. StandardScaler обязателен - без него признаки с большим размахом задавят остальные при подсчёте расстояния.

In [9]:
knn_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    KNeighborsRegressor(),
)

knn_params = {
    "kneighborsregressor__n_neighbors": [3, 5, 7, 10, 15, 20],
    "kneighborsregressor__weights": ["uniform", "distance"],
}

knn_search = GridSearchCV(
    knn_pipe, knn_params,
    cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1,
)
knn_search.fit(X_train, y_train_log, groups=groups_train)

knn_cv_results = pd.DataFrame({
    "n_neighbors": knn_search.cv_results_["param_kneighborsregressor__n_neighbors"],
    "weights": knn_search.cv_results_["param_kneighborsregressor__weights"],
    "CV MAE (лог-шкала)": -knn_search.cv_results_["mean_test_score"],
})
display(knn_cv_results.sort_values("CV MAE (лог-шкала)"))

knn_best = knn_search.best_estimator_
print(f"Лучшие параметры: {knn_search.best_params_}")
print(f"CV MAE (лог-шкала): {-knn_search.best_score_:.4f}")

,n_neighbors,weights,CV MAE (лог-шкала)
5,7,distance,0.862027
7,10,distance,0.865977
3,5,distance,0.874115
1,3,distance,0.883106
0,3,uniform,0.891149
4,7,uniform,0.892819
2,5,uniform,0.897064
9,15,distance,0.901354
6,10,uniform,0.902140
11,20,distance,0.942308


Лучшие параметры: {'kneighborsregressor__n_neighbors': 7, 'kneighborsregressor__weights': 'distance'}
CV MAE (лог-шкала): 0.8620


Посмотрим на тесте.

In [10]:
y_pred_log = knn_best.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test_log)

knn_mae = mean_absolute_error(y_true, y_pred)
knn_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"KNN | Test MAE: {knn_mae:.2f} mM | Test RMSE: {knn_rmse:.2f} mM")

KNN | Test MAE: 328.65 mM | Test RMSE: 527.09 mM


MAE модели составляет 328.65 mM. KNN заметно превосходит обе линейные модели, поэтому зависимость CC50 от дескрипторов, вероятно, не является полностью линейной.

## 8. Случайный лес

Ансамбль деревьев, каждое на своей подвыборке объектов и признаков. Итог - среднее всех деревьев. Устойчив к шуму, не требует масштабирования, ловит нелинейные зависимости.

In [11]:
rf_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestRegressor(random_state=42),
)

rf_params = {
    "randomforestregressor__n_estimators": [100, 200, 300],
    "randomforestregressor__max_depth": [10, 20, 30, None],
    "randomforestregressor__min_samples_leaf": [1, 3, 5],
}

rf_search = RandomizedSearchCV(
    rf_pipe, rf_params,
    n_iter=20, cv=cv, scoring="neg_mean_absolute_error",
    n_jobs=-1, random_state=42,
)
rf_search.fit(X_train, y_train_log, groups=groups_train)

rf_cv_results = pd.DataFrame({
    "n_estimators": rf_search.cv_results_["param_randomforestregressor__n_estimators"],
    "max_depth": rf_search.cv_results_["param_randomforestregressor__max_depth"].astype(str),
    "min_samples_leaf": rf_search.cv_results_["param_randomforestregressor__min_samples_leaf"],
    "CV MAE (лог-шкала)": -rf_search.cv_results_["mean_test_score"],
})
display(rf_cv_results.sort_values("CV MAE (лог-шкала)").head(10))

rf_best = rf_search.best_estimator_
print(f"Лучшие параметры: {rf_search.best_params_}")
print(f"CV MAE (лог-шкала): {-rf_search.best_score_:.4f}")

,n_estimators,max_depth,min_samples_leaf,CV MAE (лог-шкала)
16,200,30,1,0.885870
14,300,None,1,0.886441
5,200,None,3,0.887202
1,200,20,3,0.887349
18,300,20,1,0.887614
13,200,10,3,0.890702
19,200,10,1,0.891035
17,300,10,3,0.893840
6,100,30,3,0.894621
3,100,None,3,0.894621


Лучшие параметры: {'randomforestregressor__n_estimators': 200, 'randomforestregressor__min_samples_leaf': 1, 'randomforestregressor__max_depth': 30}
CV MAE (лог-шкала): 0.8859


Посмотрим на тесте.

In [12]:
y_pred_log = rf_best.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test_log)

rf_mae = mean_absolute_error(y_true, y_pred)
rf_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Случайный лес | Test MAE: {rf_mae:.2f} mM | Test RMSE: {rf_rmse:.2f} mM")

Случайный лес | Test MAE: 315.93 mM | Test RMSE: 504.86 mM


MAE случайного леса составляет 315.93 mM — это лучший результат на данном этапе. Модель заметно превосходит Ridge и Lasso, что может указывать на наличие нелинейных зависимостей.

## 9. Градиентный бустинг

Деревья строятся последовательно, каждое исправляет ошибки предыдущих. Обычно даёт лучшее качество, но чувствительнее к гиперпараметрам, чем случайный лес.

In [13]:
gb_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    GradientBoostingRegressor(random_state=42),
)

gb_params = {
    "gradientboostingregressor__n_estimators": [100, 200, 300],
    "gradientboostingregressor__max_depth": [3, 5, 7],
    "gradientboostingregressor__learning_rate": [0.01, 0.05, 0.1],
}

gb_search = RandomizedSearchCV(
    gb_pipe, gb_params,
    n_iter=20, cv=cv, scoring="neg_mean_absolute_error",
    n_jobs=-1, random_state=42,
)
gb_search.fit(X_train, y_train_log, groups=groups_train)

gb_cv_results = pd.DataFrame({
    "n_estimators": gb_search.cv_results_["param_gradientboostingregressor__n_estimators"],
    "max_depth": gb_search.cv_results_["param_gradientboostingregressor__max_depth"],
    "learning_rate": gb_search.cv_results_["param_gradientboostingregressor__learning_rate"],
    "CV MAE (лог-шкала)": -gb_search.cv_results_["mean_test_score"],
})
display(gb_cv_results.sort_values("CV MAE (лог-шкала)").head(10))

gb_best = gb_search.best_estimator_
print(f"Лучшие параметры: {gb_search.best_params_}")
print(f"CV MAE (лог-шкала): {-gb_search.best_score_:.4f}")

,n_estimators,max_depth,learning_rate,CV MAE (лог-шкала)
1,200,5,0.05,0.892218
8,100,5,0.05,0.903420
18,300,5,0.10,0.906863
15,200,5,0.10,0.907076
7,300,7,0.05,0.909457
6,200,7,0.05,0.909832
3,100,5,0.10,0.911119
14,100,7,0.05,0.911695
5,300,3,0.05,0.916504
17,200,7,0.10,0.920582


Лучшие параметры: {'gradientboostingregressor__n_estimators': 200, 'gradientboostingregressor__max_depth': 5, 'gradientboostingregressor__learning_rate': 0.05}
CV MAE (лог-шкала): 0.8922


Посмотрим на тесте.

In [14]:
y_pred_log = gb_best.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test_log)

gb_mae = mean_absolute_error(y_true, y_pred)
gb_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Градиентный бустинг | Test MAE: {gb_mae:.2f} mM | Test RMSE: {gb_rmse:.2f} mM")

Градиентный бустинг | Test MAE: 317.48 mM | Test RMSE: 516.45 mM


MAE градиентного бустинга составляет 317.48 mM. Результат близок к случайному лесу: разница между моделями составляет только 1.55 mM.

## 10. Сравнение

In [15]:
results = pd.DataFrame({
    "Модель": [
        "Baseline",
        "Ridge",
        "Lasso",
        "KNN",
        "Случайный лес",
        "Градиентный бустинг",
    ],
    "MAE, mM": [
        dummy_mae,
        ridge_mae,
        lasso_mae,
        knn_mae,
        rf_mae,
        gb_mae,
    ],
    "RMSE, mM": [
        dummy_rmse,
        ridge_rmse,
        lasso_rmse,
        knn_rmse,
        rf_rmse,
        gb_rmse,
    ],
})

results = results.sort_values("MAE, mM").reset_index(drop=True)
display(results.round(2))

,Модель,"MAE, mM","RMSE, mM"
0,Случайный лес,315.93,504.86
1,Градиентный бустинг,317.48,516.45
2,KNN,328.65,527.09
3,Lasso,385.13,624.28
4,Ridge,389.20,638.71
5,Baseline,475.62,694.26


Все обученные модели превзошли baseline с MAE 475.62 mM. Это показывает, что молекулярные дескрипторы содержат полезную информацию для прогнозирования CC50.

Случайный лес показал лучший результат на тестовой выборке: MAE 315.93 mM и RMSE 504.86 mM. Градиентный бустинг получил близкое значение MAE 317.48 mM, поэтому по одному разбиению нельзя считать одну из этих моделей однозначно лучше.

KNN занял третье место с MAE 328.65 mM и также заметно превзошёл линейные модели. Ridge и Lasso показали наиболее высокие ошибки: 389.20 и 385.13 mM соответственно.

Результаты показывают, что нелинейные модели лучше описывают зависимость CC50 от молекулярных дескрипторов. При этом RMSE у всех моделей заметно превышает MAE, следовательно, в данных остаются объекты с крупными ошибками прогноза.

Lasso занулил 120 коэффициентов из 210, но из-за корреляции дескрипторов это нельзя интерпретировать как окончательный отбор ненужных признаков.

Групповое разбиение исключает попадание объектов с одинаковыми дескрипторами одновременно в обучение и тест, поэтому оценка качества является более корректной.